In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, TimeSeriesSplit
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import joblib

In [2]:
path = r'C:\Users\devan\Desktop\SAR_Work\final_df.csv'
df = pd.read_csv(path, index_col=0, parse_dates=True)
df.head()

,HOURLY_KWH,AVG_CURRENT,AVG_V_LN,original_data,power_proxy,hour,weekday,month,week_of_year,hour_sin,...,kwh_roll_3h_mean,kwh_roll_24h_mean,kwh_roll_24h_std,kwh_roll_24h_min,kwh_roll_24h_max,kwh_roll_168h_mean,kwh_roll_168h_std,kwh_ratio_to_24h_avg,kwh_ratio_to_168h_avg,Type
Time,,,,,,,,,,,,,,,,,,,,,
2025-07-29 20:00:00,2.911,4.934083,231.100833,True,1140.270770,20,1,7,31,-0.866025,...,2.701000,2.795417,0.225463,2.228,3.162,2.294788,0.665732,0.869637,1.059357,YWNC2 CONE
2025-07-29 21:00:00,3.162,5.462750,229.131000,True,1251.685370,21,1,7,31,-0.707107,...,2.750000,2.802500,0.226347,2.228,3.162,2.311784,0.644555,1.038715,1.259200,YWNC2 CONE
2025-07-29 22:00:00,2.629,5.446273,230.278455,True,1254.159267,22,1,7,31,-0.500000,...,2.834667,2.805042,0.230155,2.228,3.162,2.327682,0.632400,1.127256,1.358432,YWNC2 CONE
2025-07-29 23:00:00,2.672,4.424583,230.528333,True,1019.991822,23,1,7,31,-0.258819,...,2.900667,2.793417,0.231769,2.228,3.162,2.336257,0.626659,0.941141,1.125304,YWNC2 CONE
2025-07-30 00:00:00,2.694,5.329182,230.007636,True,1225.752514,0,2,7,31,0.000000,...,2.821000,2.799958,0.225706,2.228,3.162,2.344820,0.621364,0.954300,1.139533,YWNC2 CONE


In [4]:
df3cone = df[df["Type"] == "YWNC3 CONE"].drop(columns="Type")
df2cone = df[df["Type"] == "YWNC2 CONE"].drop(columns="Type")
df3cup = df[df["Type"] == "YWNC3 CUP"].drop(columns="Type")
df2cup = df[df["Type"] == "YWNC2 CUP"].drop(columns="Type")

In [6]:
df2cone.head()

,HOURLY_KWH,AVG_CURRENT,AVG_V_LN,original_data,power_proxy,hour,weekday,month,week_of_year,hour_sin,...,kwh_lag_168,kwh_roll_3h_mean,kwh_roll_24h_mean,kwh_roll_24h_std,kwh_roll_24h_min,kwh_roll_24h_max,kwh_roll_168h_mean,kwh_roll_168h_std,kwh_ratio_to_24h_avg,kwh_ratio_to_168h_avg
Time,,,,,,,,,,,,,,,,,,,,,
2025-07-29 20:00:00,2.911,4.934083,231.100833,True,1140.270770,20,1,7,31,-0.866025,...,0.072714,2.701000,2.795417,0.225463,2.228,3.162,2.294788,0.665732,0.869637,1.059357
2025-07-29 21:00:00,3.162,5.462750,229.131000,True,1251.685370,21,1,7,31,-0.707107,...,0.507000,2.750000,2.802500,0.226347,2.228,3.162,2.311784,0.644555,1.038715,1.259200
2025-07-29 22:00:00,2.629,5.446273,230.278455,True,1254.159267,22,1,7,31,-0.500000,...,1.197000,2.834667,2.805042,0.230155,2.228,3.162,2.327682,0.632400,1.127256,1.358432
2025-07-29 23:00:00,2.672,4.424583,230.528333,True,1019.991822,23,1,7,31,-0.258819,...,1.242000,2.900667,2.793417,0.231769,2.228,3.162,2.336257,0.626659,0.941141,1.125304
2025-07-30 00:00:00,2.694,5.329182,230.007636,True,1225.752514,0,2,7,31,0.000000,...,1.901000,2.821000,2.799958,0.225706,2.228,3.162,2.344820,0.621364,0.954300,1.139533


In [15]:
df2cone_9_to_15_feb = df2cone.loc["2026-02-09 07:00:00":"2026-02-16 06:00:00"]
df3cone_9_to_15_feb = df3cone.loc["2026-02-09 07:00:00":"2026-02-16 06:00:00"]
df2cup_9_to_15_feb = df2cup.loc["2026-02-09 07:00:00":"2026-02-16 06:00:00"]
df3cup_9_to_15_feb = df3cup.loc["2026-02-09 07:00:00":"2026-02-16 06:00:00"]

In [16]:
df2cone_9_to_15_feb["HOURLY_KWH"]

Time
2026-02-09 07:00:00    1.028
2026-02-09 08:00:00    1.605
2026-02-09 09:00:00    1.506
2026-02-09 10:00:00    1.595
2026-02-09 11:00:00    1.498
                       ...  
2026-02-16 02:00:00    0.148
2026-02-16 03:00:00    0.143
2026-02-16 04:00:00    0.149
2026-02-16 05:00:00    0.142
2026-02-16 06:00:00    0.319
Name: HOURLY_KWH, Length: 168, dtype: float64

In [17]:
daily_kwh_2cone = df2cone_9_to_15_feb["HOURLY_KWH"].resample("D").sum()
daily_kwh_3cone = df3cone_9_to_15_feb["HOURLY_KWH"].resample("D").sum()
daily_kwh_2cup = df2cup_9_to_15_feb["HOURLY_KWH"].resample("D").sum()
daily_kwh_3cup = df3cup_9_to_15_feb["HOURLY_KWH"].resample("D").sum()

In [20]:
print(daily_kwh_2cone)
print("_"*100)
print(daily_kwh_3cone)
print("_"*100)
print(daily_kwh_2cup)
print("_"*100)
print(daily_kwh_3cup)
print("_"*100)

Time
2026-02-09    22.027000
2026-02-10    31.073000
2026-02-11    29.784982
2026-02-12    21.404000
2026-02-13    29.689000
2026-02-14    29.866000
2026-02-15    19.130000
2026-02-16     1.194000
Freq: D, Name: HOURLY_KWH, dtype: float64
____________________________________________________________________________________________________
Time
2026-02-09    13.925000
2026-02-10    19.464000
2026-02-11    18.529749
2026-02-12    14.041000
2026-02-13    19.627000
2026-02-14    18.773000
2026-02-15    14.717000
2026-02-16     0.826000
Freq: D, Name: HOURLY_KWH, dtype: float64
____________________________________________________________________________________________________
Time
2026-02-09    14.879000
2026-02-10    42.520000
2026-02-11    41.321815
2026-02-12    27.198000
2026-02-13    32.640000
2026-02-14    31.184000
2026-02-15    24.293000
2026-02-16     1.044000
Freq: D, Name: HOURLY_KWH, dtype: float64
_________________________________________________________________________________